In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import sys
sys.path.append('../src')

# Load cleaned ratings
ratings = pd.read_csv('../data/processed/ratings_clean.csv')
movies = pd.read_csv('../data/processed/movies_clean.csv')

print(f"Total ratings: {len(ratings):,}")
print(f"Total users: {ratings['userId'].nunique():,}")
print(f"Total movies: {ratings['movieId'].nunique():,}")

# Fast vectorized split — same result as per-user loop, fraction of the time
np.random.seed(42)
ratings['_random'] = np.random.rand(len(ratings))
ratings['_rank'] = ratings.groupby('userId')['_random'].rank(pct=True)

train_df = ratings[ratings['_rank'] <= 0.8].drop(columns=['_random', '_rank']).reset_index(drop=True)
test_df  = ratings[ratings['_rank'] >  0.8].drop(columns=['_random', '_rank']).reset_index(drop=True)

print(f"\nTraining set: {len(train_df):,} ratings ({len(train_df)/len(ratings)*100:.1f}%)")
print(f"Test set:     {len(test_df):,} ratings ({len(test_df)/len(ratings)*100:.1f}%)")
print(f"Users in training: {train_df['userId'].nunique():,}")
print(f"Users in test:     {test_df['userId'].nunique():,}")

# Leakage check
overlap = pd.merge(
    train_df[['userId','movieId']],
    test_df[['userId','movieId']],
    on=['userId','movieId']
)
print(f"\nData leakage check — overlapping pairs: {len(overlap):,}")
print("(Must be 0)")

Total ratings: 20,000,263
Total users: 138,493
Total movies: 26,744

Training set: 15,945,812 ratings (79.7%)
Test set:     4,054,451 ratings (20.3%)
Users in training: 138,493
Users in test:     138,493

Data leakage check — overlapping pairs: 0
(Must be 0)


In [2]:
import joblib
from scipy.sparse import csr_matrix
from collaborative_filtering import train_svd

# Step 1: Build sparse matrix from TRAINING DATA ONLY
print("Building sparse matrix from training data...")

# Recreate ID mappings from training data only
train_users = train_df['userId'].unique()
train_movies = train_df['movieId'].unique()

user_to_idx_train = {uid: idx for idx, uid in enumerate(train_users)}
movie_to_idx_train = {mid: idx for idx, mid in enumerate(train_movies)}
idx_to_movie_train = {idx: mid for mid, idx in movie_to_idx_train.items()}

row_indices = train_df['userId'].map(user_to_idx_train)
col_indices = train_df['movieId'].map(movie_to_idx_train)

sparse_train = csr_matrix(
    (train_df['rating'], (row_indices, col_indices)),
    shape=(len(train_users), len(train_movies))
)
print(f"Training sparse matrix: {sparse_train.shape}")

# Step 2: Train SVD on training data only
print("\nTraining SVD on training data...")
svd_eval, user_factors_eval = train_svd(sparse_train, n_components=50)
print(f"Done. Variance explained: {svd_eval.explained_variance_ratio_.sum()*100:.2f}%")

Building sparse matrix from training data...
Training sparse matrix: (138493, 25857)

Training SVD on training data...
Done. Variance explained: 30.10%


In [3]:
from sklearn.metrics import mean_squared_error
import numpy as np

print("Calculating predictions for test set...")

# Only evaluate on test ratings where the movie EXISTS in training data
# (we can't predict for movies the model never saw)
test_known = test_df[
    test_df['userId'].isin(user_to_idx_train) &
    test_df['movieId'].isin(movie_to_idx_train)
].copy()

print(f"Test ratings with known users AND movies: {len(test_known):,}")
print(f"Test ratings skipped (unseen movies): {len(test_df) - len(test_known):,}")

# Predict ratings for each test entry
actuals = []
predictions = []

for _, row in test_known.iterrows():
    user_idx = user_to_idx_train[row['userId']]
    movie_idx = movie_to_idx_train[row['movieId']]

    # Predict: dot product of this user's factors with this movie's factors
    predicted = np.dot(
        user_factors_eval[user_idx],
        svd_eval.components_[:, movie_idx]
    )

    actuals.append(row['rating'])
    predictions.append(predicted)

actuals = np.array(actuals)
predictions = np.array(predictions)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(actuals, predictions))
print(f"\nRMSE: {rmse:.4f}")
print(f"Interpretation: on average, predictions are off by {rmse:.2f} stars")

Calculating predictions for test set...
Test ratings with known users AND movies: 4,053,442
Test ratings skipped (unseen movies): 1,009

RMSE: 2.8060
Interpretation: on average, predictions are off by 2.81 stars


In [4]:
from sklearn.metrics import mean_squared_error
import numpy as np

print("Calculating RMSE (vectorized, with proper clipping)...")

# Filter to known entries
test_known = test_df[
    test_df['userId'].isin(user_to_idx_train) &
    test_df['movieId'].isin(movie_to_idx_train)
].copy()

# Vectorized prediction — no loop needed
test_known['user_idx'] = test_known['userId'].map(user_to_idx_train)
test_known['movie_idx'] = test_known['movieId'].map(movie_to_idx_train)

user_vecs  = user_factors_eval[test_known['user_idx'].values]
movie_vecs = svd_eval.components_[:, test_known['movie_idx'].values].T

raw_predictions = np.sum(user_vecs * movie_vecs, axis=1)

# Clip to valid rating range (0.5 - 5.0)
predictions_clipped = np.clip(raw_predictions, 0.5, 5.0)
actuals = test_known['rating'].values

# Show what clipping actually did
print(f"Raw predictions range: {raw_predictions.min():.2f} to {raw_predictions.max():.2f}")
print(f"After clipping:        {predictions_clipped.min():.2f} to {predictions_clipped.max():.2f}")

# RMSE and MAE with clipped predictions
rmse = np.sqrt(mean_squared_error(actuals, predictions_clipped))
mae  = np.mean(np.abs(actuals - predictions_clipped))

print(f"\n--- Results ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"\nInterpretation:")
print(f"  RMSE {rmse:.2f} → large errors penalized more heavily")
print(f"  MAE  {mae:.2f} → average absolute error across all predictions")

# Baseline comparison: what if we just predicted the mean rating every time?
mean_rating = actuals.mean()
baseline_rmse = np.sqrt(np.mean((actuals - mean_rating)**2))
baseline_mae  = np.mean(np.abs(actuals - mean_rating))
print(f"\n--- Baseline (predict mean {mean_rating:.2f} for everything) ---")
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Baseline MAE:  {baseline_mae:.4f}")
print(f"\nSVD improvement over baseline:")
print(f"  RMSE improvement: {baseline_rmse - rmse:.4f} ({(baseline_rmse-rmse)/baseline_rmse*100:.1f}%)")
print(f"  MAE improvement:  {baseline_mae - mae:.4f}  ({(baseline_mae-mae)/baseline_mae*100:.1f}%)")

Calculating RMSE (vectorized, with proper clipping)...
Raw predictions range: -5.25 to 9.39
After clipping:        0.50 to 5.00

--- Results ---
RMSE: 2.6827
MAE:  2.4357

Interpretation:
  RMSE 2.68 → large errors penalized more heavily
  MAE  2.44 → average absolute error across all predictions

--- Baseline (predict mean 3.53 for everything) ---
Baseline RMSE: 1.0522
Baseline MAE:  0.8413

SVD improvement over baseline:
  RMSE improvement: -1.6304 (-154.9%)
  MAE improvement:  -1.5944  (-189.5%)


In [5]:
from scipy.sparse import csr_matrix
from collaborative_filtering import train_svd
import numpy as np
from sklearn.metrics import mean_squared_error

print("Rebuilding with mean-centered ratings...")

# Step 1: Calculate each user's mean rating from TRAINING DATA only
user_means = train_df.groupby('userId')['rating'].mean()

# Step 2: Subtract each user's mean from their ratings (mean-centering)
train_centered = train_df.copy()
train_centered['rating_centered'] = (
    train_df['rating'] - train_df['userId'].map(user_means)
)

print(f"Original rating range: {train_df['rating'].min()} to {train_df['rating'].max()}")
print(f"Centered rating range: {train_centered['rating_centered'].min():.2f} to {train_centered['rating_centered'].max():.2f}")

# Step 3: Rebuild ID mappings
train_users = train_df['userId'].unique()
train_movies = train_df['movieId'].unique()
user_to_idx_train = {uid: idx for idx, uid in enumerate(train_users)}
movie_to_idx_train = {mid: idx for idx, mid in enumerate(train_movies)}

# Step 4: Build sparse matrix with CENTERED ratings
row_indices = train_centered['userId'].map(user_to_idx_train)
col_indices = train_centered['movieId'].map(movie_to_idx_train)

sparse_train_centered = csr_matrix(
    (train_centered['rating_centered'], (row_indices, col_indices)),
    shape=(len(train_users), len(train_movies))
)
print(f"\nCentered sparse matrix shape: {sparse_train_centered.shape}")

# Step 5: Train SVD on centered matrix
print("Training SVD on centered matrix...")
svd_centered, user_factors_centered = train_svd(
    sparse_train_centered, n_components=50
)
print(f"Done. Variance explained: {svd_centered.explained_variance_ratio_.sum()*100:.2f}%")

Rebuilding with mean-centered ratings...
Original rating range: 0.5 to 5.0
Centered rating range: -4.38 to 4.22

Centered sparse matrix shape: (138493, 25857)
Training SVD on centered matrix...
Done. Variance explained: 16.23%


In [6]:
print("Calculating RMSE with mean-centered SVD...")

# Filter test set to known users and movies
test_known = test_df[
    test_df['userId'].isin(user_to_idx_train) &
    test_df['movieId'].isin(movie_to_idx_train)
].copy()

print(f"Test ratings evaluated: {len(test_known):,}")

# Map to matrix indices
test_known['user_idx'] = test_known['userId'].map(user_to_idx_train)
test_known['movie_idx'] = test_known['movieId'].map(movie_to_idx_train)

# Vectorized prediction
user_vecs  = user_factors_centered[test_known['user_idx'].values]
movie_vecs = svd_centered.components_[:, test_known['movie_idx'].values].T

# Raw SVD output (centered prediction)
raw_centered_predictions = np.sum(user_vecs * movie_vecs, axis=1)

# Add each user's mean back to get actual rating predictions
user_mean_values = test_known['userId'].map(user_means).values
final_predictions = raw_centered_predictions + user_mean_values

# Clip to valid range
final_predictions_clipped = np.clip(final_predictions, 0.5, 5.0)
actuals = test_known['rating'].values

print(f"Raw centered range:  {raw_centered_predictions.min():.2f} to {raw_centered_predictions.max():.2f}")
print(f"After adding mean:   {final_predictions.min():.2f} to {final_predictions.max():.2f}")
print(f"After clipping:      {final_predictions_clipped.min():.2f} to {final_predictions_clipped.max():.2f}")

# Calculate metrics
rmse = np.sqrt(mean_squared_error(actuals, final_predictions_clipped))
mae  = np.mean(np.abs(actuals - final_predictions_clipped))

print(f"\n--- SVD Results (mean-centered) ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

# Baseline comparison
mean_rating = actuals.mean()
baseline_rmse = np.sqrt(np.mean((actuals - mean_rating)**2))
baseline_mae  = np.mean(np.abs(actuals - mean_rating))

print(f"\n--- Baseline (predict mean {mean_rating:.2f} for everything) ---")
print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Baseline MAE:  {baseline_mae:.4f}")

print(f"\n--- SVD vs Baseline ---")
print(f"RMSE improvement: {baseline_rmse - rmse:.4f} ({(baseline_rmse-rmse)/baseline_rmse*100:.1f}%)")
print(f"MAE improvement:  {baseline_mae  - mae:.4f}  ({(baseline_mae-mae)/baseline_mae*100:.1f}%)")

better = "✅ SVD beats baseline" if rmse < baseline_rmse else "❌ SVD worse than baseline"
print(f"\n{better}")

Calculating RMSE with mean-centered SVD...
Test ratings evaluated: 4,053,442
Raw centered range:  -3.06 to 3.61
After adding mean:   -0.23 to 6.45
After clipping:      0.50 to 5.00

--- SVD Results (mean-centered) ---
RMSE: 0.9057
MAE:  0.6969

--- Baseline (predict mean 3.53 for everything) ---
Baseline RMSE: 1.0522
Baseline MAE:  0.8413

--- SVD vs Baseline ---
RMSE improvement: 0.1466 (13.9%)
MAE improvement:  0.1443  (17.2%)

✅ SVD beats baseline


In [7]:
print("Calculating Precision@K and Recall@K for all three models...")

# Define what "relevant" means
RELEVANCE_THRESHOLD = 4.0
K_VALUES = [5, 10]

# We'll evaluate on a sample of users for speed
# (full 138K users would take too long)
np.random.seed(42)
eval_users = np.random.choice(
    test_df['userId'].unique(), size=500, replace=False
)

print(f"Evaluating on {len(eval_users)} sampled users...")

# Load what we need for all three models
from popularity_recommender import build_popularity_table, recommend_popular
from content_based_recommender import build_combined_similarity, recommend_similar_movies
from scipy.sparse import load_npz
import joblib

# Load pre-built features
genres_encoded = pd.read_csv('../data/features/genres_encoded.csv')
tfidf_matrix   = load_npz('../data/features/tfidf_matrix.npz')
tfidf_movie_ids = pd.read_csv('../data/features/tfidf_movieId_index.csv')['movieId']

# Build popularity table from training data only
popularity_table = build_popularity_table(train_df, movies, min_ratings=100)

# Build content similarity matrix
print("Building content similarity matrix...")
combined_sim = build_combined_similarity(
    genres_df=genres_encoded,
    tfidf_matrix=tfidf_matrix,
    tfidf_movie_ids=tfidf_movie_ids,
    genre_weight=0.3,
    tag_weight=0.7
)
print("Done.")

Calculating Precision@K and Recall@K for all three models...
Evaluating on 500 sampled users...
Building content similarity matrix...


MemoryError: Unable to allocate 5.54 GiB for an array with shape (27278, 27278) and data type float64

In [8]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import numpy as np

print("Setting up lightweight content-based evaluation...")

# Load genre features (small — 27278 x 19, fine in memory)
genres_encoded = pd.read_csv('../data/features/genres_encoded.csv')
genre_features = genres_encoded.drop(columns=['movieId']).values
genre_movie_ids = genres_encoded['movieId'].values

# Create a movieId → row index lookup for genre matrix
genre_idx = {mid: idx for idx, mid in enumerate(genre_movie_ids)}

def get_content_recs(liked_movie_ids, n=10, exclude_ids=None):
    """
    Given a list of movies the user liked, find the top N
    most similar movies using genre cosine similarity.
    Computes similarity only for the needed rows — no full matrix.
    """
    if exclude_ids is None:
        exclude_ids = set()

    # Get genre vectors for liked movies
    liked_indices = [genre_idx[mid] for mid in liked_movie_ids
                     if mid in genre_idx]
    if not liked_indices:
        return []

    liked_vecs = genre_features[liked_indices]  # shape: (n_liked, 19)

    # Compute similarity between liked movies and ALL movies
    # This is a small operation: n_liked × 27278, not 27278 × 27278
    sims = cos_sim(liked_vecs, genre_features)  # shape: (n_liked, 27278)

    # Average similarity across all liked movies
    avg_sims = sims.mean(axis=0)  # shape: (27278,)

    # Build scored list, excluding already-seen movies
    scores = [
        (genre_movie_ids[i], avg_sims[i])
        for i in range(len(genre_movie_ids))
        if genre_movie_ids[i] not in exclude_ids
    ]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [mid for mid, _ in scores[:n]]

print("Content-based function ready (on-demand, memory safe)")
print("Now calculating Precision@K and Recall@K for all three models...")

RELEVANCE_THRESHOLD = 4.0
K_VALUES = [5, 10]

np.random.seed(42)
eval_users = np.random.choice(
    test_df['userId'].unique(), size=500, replace=False
)

# Storage for results
results = {
    'popularity': {k: {'precision': [], 'recall': []} for k in K_VALUES},
    'content':    {k: {'precision': [], 'recall': []} for k in K_VALUES},
    'svd':        {k: {'precision': [], 'recall': []} for k in K_VALUES},
}

# Popularity recommendations are the same for every user
pop_recs = {k: list(popularity_table.head(k)['movieId']) for k in K_VALUES}

for i, user_id in enumerate(eval_users):
    if i % 100 == 0:
        print(f"  Processing user {i+1}/500...")

    # Get this user's test ratings
    user_test = test_df[test_df['userId'] == user_id]
    if len(user_test) == 0:
        continue

    # Ground truth: movies this user actually liked in the test set
    relevant_movies = set(
        user_test[user_test['rating'] >= RELEVANCE_THRESHOLD]['movieId']
    )
    if len(relevant_movies) == 0:
        continue

    # Movies this user rated in TRAINING (already seen — exclude from recs)
    user_train_movies = set(
        train_df[train_df['userId'] == user_id]['movieId']
    )

    # Movies this user liked in training (seed for content-based)
    user_liked_train = list(
        train_df[
            (train_df['userId'] == user_id) &
            (train_df['rating'] >= RELEVANCE_THRESHOLD)
        ]['movieId']
    )

    for k in K_VALUES:
        # --- Popularity recommendations ---
        pop_k = [m for m in pop_recs[k] if m not in user_train_movies][:k]

        # --- Content-based recommendations ---
        if user_liked_train:
            content_k = get_content_recs(
                user_liked_train, n=k,
                exclude_ids=user_train_movies
            )
        else:
            content_k = pop_k  # fallback to popularity if no liked history

        # --- SVD recommendations ---
        if user_id in user_to_idx_train:
            user_idx = user_to_idx_train[user_id]
            user_vec = user_factors_centered[user_idx]
            scores = np.dot(user_vec, svd_centered.components_)
            # Add user mean back
            user_mean = user_means.get(user_id, actuals.mean())
            scores = scores + user_mean
            # Exclude already-seen movies
            movie_id_array = np.array(
                [movie_to_idx_train.get(mid, -1)
                 for mid in genre_movie_ids]
            )
            scored = []
            for idx, mid in enumerate(genre_movie_ids):
                if mid in user_train_movies:
                    continue
                m_idx = movie_to_idx_train.get(mid, -1)
                if m_idx == -1:
                    continue
                scored.append((mid, scores[m_idx]))
            scored.sort(key=lambda x: x[1], reverse=True)
            svd_k = [mid for mid, _ in scored[:k]]
        else:
            svd_k = pop_k  # fallback

        # --- Calculate metrics ---
        for model, recs in [('popularity', pop_k),
                             ('content', content_k),
                             ('svd', svd_k)]:
            hits = len(set(recs) & relevant_movies)
            precision = hits / k if k > 0 else 0
            recall = hits / len(relevant_movies) if relevant_movies else 0
            results[model][k]['precision'].append(precision)
            results[model][k]['recall'].append(recall)

# Print results
print("\n" + "="*55)
print(f"{'Model':<15} {'P@5':>8} {'P@10':>8} {'R@5':>8} {'R@10':>8}")
print("="*55)
for model in ['popularity', 'content', 'svd']:
    p5  = np.mean(results[model][5]['precision'])
    p10 = np.mean(results[model][10]['precision'])
    r5  = np.mean(results[model][5]['recall'])
    r10 = np.mean(results[model][10]['recall'])
    print(f"{model:<15} {p5:>8.4f} {p10:>8.4f} {r5:>8.4f} {r10:>8.4f}")
print("="*55)

Setting up lightweight content-based evaluation...
Content-based function ready (on-demand, memory safe)
Now calculating Precision@K and Recall@K for all three models...
  Processing user 1/500...
  Processing user 101/500...
  Processing user 201/500...
  Processing user 301/500...
  Processing user 401/500...

Model                P@5     P@10      R@5     R@10
popularity        0.0516   0.0350   0.0244   0.0327
content           0.0073   0.0077   0.0023   0.0045
svd               0.1858   0.1419   0.0810   0.1155
